# Nested Learning for CAD — continuation7.1 sau bridge384

Giữ nguyên method v5.1 và core source r2; chỉ thêm lớp vận hành khóa riêng. Không chạy lại W6/pilot/G3/bridge. Chỉ NESTED_seed0; chưa chạy CADIC hoặc matrix trong notebook này.

**Bốn inputs:** (1) source mới `hope_cad_support_v5_1_continuation_source_r1.zip` (hoặc Kaggle đã giải nén), (2) **reviewed bridge384 output** lần đầu; lần sau **latest continuation output** (tháo bridge/outputs cũ), (3) MVTec AD, (4) `vit_base_patch8_224.augreg_in21k.safetensors`. Không cần attach source bridge r2 riêng vì nó đã được giữ byte-exact trong source mới. Không attach paused376 hoặc pilot/W6.

GPU T4×2, kernel mới; Save Version → Save & Run All. SESSION_HOURS=9 là soft budget, trừ600s để export, không là ETA. Deadline chỉ kiểm ở atomic boundary; platform hard kill vẫn có thể mất output. Không tự đổi package/runtime. Có heartbeat30s và progress mỗi8ảnh; bootstrap/staging có log trước worker. Lần đầu384→3599, rồi15final evaluation cells; pause tiếp từ checkpoint mới, không train lại từ384. Không dùng notebook7.1 cũ để resume source này.

Chỉ `7.1_complete_pending_review` là chạy xong, vẫn cần review artifacts. `paused_at_atomic_boundary`: tải ZIP/log, input latest output rồi chạy notebook này tiếp. `failed_preserve_evidence`: giữ ZIP/log để review, không tự retry. Điểm AD không được dùng để sửa method/threshold hoặc chọn category. Chưa cho phép VisA.

Lưu output và Kaggle log vào `D:/src/DATN/.agents/artifacts/runs/support_v5_1_benchmark_NESTED_seed0_continuation/session_01/`; phiên tiếp theo session_02,03… File output `hope_cad_support_v5_1_continuation_evidence.zip`. Final metrics ở `benchmark/runs/NESTED_seed0/metrics/`. Đọc `docs/main_spec/continuation_7_1_v5_1.md` và implementation trước chạy.


In [ ]:
PHASE='7.1'
JOB='NESTED_seed0'
SESSION_HOURS=9.0
SOURCE_OVERRIDE=None
PARENT_OVERRIDE=None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

import os,sys,time,json,hashlib,zipfile,shutil
from pathlib import Path,PurePosixPath
STARTED=time.monotonic()
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
sys.dont_write_bytecode=True
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_continuation_7_1_source_r1')
WORK=Path('/kaggle/working/hope_continuation_7_1')
ASSETS=Path('/tmp/hope_continuation_assets')
OUTPUT=Path('/kaggle/working/hope_cad_support_v5_1_continuation_evidence.zip')
print('[CONTINUATION BOOTSTRAP] finding hash-bound source',flush=True)
def safe(root,name):
    p=PurePosixPath(name)
    if p.is_absolute() or p.as_posix()!=name or '..' in p.parts or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    dest=root/name
    if not dest.resolve().is_relative_to(root.resolve()):raise ValueError('Source path escapes target')
    return dest
def source_manifest(p):
    if p.is_dir():return json.loads((p/'continuation_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:
        ns=z.namelist()
        if len(ns)!=len(set(ns)):raise ValueError('Duplicate source entries')
        m=json.loads(z.read('continuation_manifest.json'))
        if set(ns)!=set(m['files'])|{'continuation_manifest.json'}:raise ValueError('Source inventory changed')
        return m
choices=[]
paths=[Path(SOURCE_OVERRIDE)] if SOURCE_OVERRIDE else [p.parent for p in INPUT.rglob('continuation_manifest.json')]+list(INPUT.rglob('*source*.zip'))
for p in paths:
    try:
        m=source_manifest(p)
        if m.get('schema')=='hope_support_v5_1_continuation_7_1_source_v1':choices.append((p,m))
    except (KeyError,zipfile.BadZipFile):
        if SOURCE_OVERRIDE:raise
if not choices or len({json.dumps(m,sort_keys=True) for p,m in choices})!=1:raise ValueError('Expected ONE continuation source identity; set SOURCE_OVERRIDE')
src,m=sorted(choices,key=lambda x:str(x[0]))[0]
if ROOT.exists():raise ValueError('Source root already exists; start a fresh kernel/session')
ROOT.mkdir(parents=True)
archive=zipfile.ZipFile(src) if src.is_file() else None
try:
    for name,h in m['files'].items():
        original=safe(src,name) if not archive else None
        if original and original.is_symlink():raise ValueError('Source symlink')
        raw=archive.read(name) if archive else original.read_bytes()
        if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Source SHA mismatch: '+name)
        dst=safe(ROOT,name);dst.parent.mkdir(parents=True,exist_ok=True);dst.write_bytes(raw)
finally:
    if archive:archive.close()
(ROOT/'continuation_manifest.json').write_text(json.dumps(m,sort_keys=True))
sys.path.insert(0,str(ROOT));os.chdir(ROOT)
print('[CONTINUATION BOOTSTRAP] verified',len(m['files']),'files; frozen r2 plus separate operational extension',flush=True)
from continuation_runtime_v1.common import verify_delivery
CORE_SOURCE,ORCHESTRATION=verify_delivery()
print('[CONTINUATION IDENTITY]',json.dumps(ORCHESTRATION),flush=True)


In [ ]:
from continuation_runtime_v1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,job=JOB,session_hours=SESSION_HOURS,started=STARTED,
    inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),
    PARENT_OVERRIDE=PARENT_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
